# Análisis Exploratorio de la Brecha Digital en el Perú (Capa Gold)

Este notebook consume directamente el dataset analítico consolidado en la **Capa Gold** de HDFS (`/data/gold/brecha_digital_consolidada.parquet`).

### Objetivos:
1. Conectar con el clúster PySpark sobre YARN y leer la tabla Gold en formato Parquet.
2. Ejecutar consultas SQL analíticas sobre la evolución trimestral de conexiones fijas por departamento y tecnología.
3. Comparar la penetración de internet por operadora y el acceso en hogares con los indicadores del INEI.
4. Generar visualizaciones clave de la brecha digital interdepartamental.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession

# Inicialización de Spark apuntando a YARN / local
spark = SparkSession.builder \
    .appName("Notebook-Analisis-Brecha-Digital-Gold") \
    .master(os.getenv("SPARK_MASTER", "yarn")) \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("SparkSession iniciada con éxito. Versión de Spark:", spark.version)

### 1. Carga de la Capa Gold desde HDFS

In [ ]:
HDFS_GOLD_PATH = "hdfs://localhost:9000/data/gold/brecha_digital_consolidada.parquet"

df_gold = spark.read.parquet(HDFS_GOLD_PATH)
df_gold.printSchema()
print("Total de registros analíticos en Capa Gold:", df_gold.count())

# Registrar vista temporal para permitir consultas SQL estándar
df_gold.createOrReplaceTempView("brecha_gold")

### 2. Consulta 1: Evolución Trimestral Nacional y Adopción de Fibra Óptica

In [ ]:
query_nacional = """
SELECT
    anio,
    trimestre,
    SUM(total_conexiones) AS conexiones_nacionales,
    SUM(conexiones_fibra_optica) AS fibra_optica_nacional,
    ROUND((SUM(conexiones_fibra_optica) / SUM(total_conexiones)) * 100, 2) AS pct_adopcion_fibra
FROM brecha_gold
GROUP BY anio, trimestre
ORDER BY anio, trimestre
"""

df_nac_spark = spark.sql(query_nacional)
df_nac_spark.show(15, truncate=False)

### 3. Consulta 2: Brecha Digital Departamental (Último Año Disponible)

In [ ]:
query_departamentos = """
WITH max_periodo AS (
    SELECT MAX(anio) AS max_anio FROM brecha_gold
)
SELECT
    g.departamento,
    SUM(g.total_conexiones) AS total_conexiones,
    SUM(g.conexiones_fibra_optica) AS conexiones_fibra,
    ROUND((SUM(g.conexiones_fibra_optica) / SUM(g.total_conexiones)) * 100, 2) AS pct_fibra,
    AVG(g.total_operadoras_activas) AS promedio_operadoras
FROM brecha_gold g
JOIN max_periodo m ON g.anio = m.max_anio
WHERE g.trimestre = 4 -- Cierre anual
GROUP BY g.departamento
ORDER BY total_conexiones DESC
"""

df_dept_spark = spark.sql(query_departamentos)
df_dept_spark.show(30, truncate=False)

### 4. Consulta 3: Integración OSIPTEL + Indicadores INEI

In [ ]:
query_integracion = """
SELECT
    anio,
    trimestre,
    departamento,
    total_conexiones,
    conexiones_fibra_optica,
    hogares_internet_pct,
    estado_integracion
FROM brecha_gold
WHERE anio >= 2020
ORDER BY anio DESC, trimestre DESC, total_conexiones DESC
"""

spark.sql(query_integracion).show(10, truncate=False)

### 5. Visualización Gráfica de la Brecha Digital (Pandas + Seaborn)

In [ ]:
# Convertir agregado departamental a Pandas para graficar
pdf_dept = df_dept_spark.toPandas()

plt.figure(figsize=(12, 8))
sns.set_theme(style="whitegrid")

barplot = sns.barplot(
    data=pdf_dept,
    x="total_conexiones",
    y="departamento",
    palette="viridis"
)

plt.title("Distribución de Conexiones de Internet Fijo por Departamento (Cierre Anual)", fontsize=14, pad=15)
plt.xlabel("Total de Conexiones Activas", fontsize=12)
plt.ylabel("Departamento", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Evolución de Fibra Óptica vs Total Conexiones
pdf_nac = df_nac_spark.toPandas()
pdf_nac["periodo_label"] = pdf_nac["anio"].astype(str) + "-T" + pdf_nac["trimestre"].astype(str)

plt.figure(figsize=(14, 6))
plt.plot(pdf_nac["periodo_label"], pdf_nac["conexiones_nacionales"], marker='o', label="Total Conexiones", color="#1f77b4")
plt.plot(pdf_nac["periodo_label"], pdf_nac["fibra_optica_nacional"], marker='s', label="Fibra Óptica", color="#2ca02c")

plt.xticks(rotation=90)
plt.title("Evolución Trimestral de Conexiones Fijas y Penetración de Fibra Óptica en Perú", fontsize=14)
plt.xlabel("Periodo Trimestral", fontsize=12)
plt.ylabel("Número de Conexiones", fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()